# INFERENCE HELPERS

In [ ]:
import torch
import re
import chess

UCI_RE = re.compile(r"^[a-h][1-8][a-h][1-8][qrbn]?$")

def is_valid_uci(move: str) -> bool:
    return bool(UCI_RE.match(move))

def board_from_history(moves: list) -> chess.Board:
    board = chess.Board()
    for m in moves:
        board.push_uci(m)
    return board


def build_prompt(
    move_history: list,
    elo_group:    str = "1500-1700",
    phase:        str = "Opening",
    style:        str = "Balanced",   
    error_level:  str = "Good",
) -> str:
    """
    Build the conditioning prompt for next-move generation.

    style options: Tactical | Aggressive | Solid | Defensive | Balanced
    error_level  : Good | Inaccuracy | Mistake | Blunder

    Example usage:
      build_prompt(moves, style="Tactical", error_level="Blunder")
      → model generates a Tactical-flavoured blunder
    """
    ctx = " ".join(move_history)
    return (
        f"<ELO:{elo_group}> <PHASE:{phase}> <STYLE:{style}> <ERR:{error_level}> "
        f"<|moves|> {ctx} <|target|>"
    )


def predict_next_move(
    move_history: list,
    elo_group:    str   = "1500-1700",
    phase:        str   = "Opening",
    style:        str   = "Balanced",  
    error_level:  str   = "Good",
    temperature:  float = 1.0,
    top_p:        float = 0.95,
    max_move_tokens: int = 6,
    enforce_legal: bool  = True,
    max_retries:   int   = 5,
) -> str:
    device = next(model.parameters()).device
    model.eval()

    prompt = build_prompt(move_history, elo_group, phase, style, error_level)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[1]

    try:
        board = board_from_history(move_history)
    except Exception:
        board = None

    for attempt in range(max_retries):
        t = min(temperature * (1 + 0.3 * attempt), 3.0)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_move_tokens,
                do_sample=(t > 0.01),
                temperature=max(t, 0.01),
                top_p=top_p,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        new_tokens = output[0][input_length:]
        generated  = tokenizer.decode(new_tokens, skip_special_tokens=True)
        generated_strip = generated.strip()
        move = ""
        for length in (5, 4):
            candidate = generated_strip[:length].strip()
            if is_valid_uci(candidate):
                move = candidate
                break
        if not move:
            for token in generated_strip.split():
                if is_valid_uci(token):
                    move = token
                    break

        if not is_valid_uci(move):
            continue

        if enforce_legal and board is not None:
            try:
                if chess.Move.from_uci(move) not in board.legal_moves:
                    continue
            except Exception:
                continue

        return move

    if enforce_legal and board is not None:
        legal = list(board.legal_moves)
        if legal:
            import random
            return legal[random.randint(0, len(legal) - 1)].uci()
    return ""


# Quick test: same position, all style × error combinations
history = ["e2e4", "d7d6", "d2d4"]
STYLES  = ["Tactical", "Aggressive", "Solid", "Defensive", "Balanced"]
TEMPS = {
    "Good":       0.1, 
    "Inaccuracy": 0.8,  
    "Mistake":    1.2, 
    "Blunder":    1.5,  
}

print("Testing style × error conditioning:")
print(f"  History: {' '.join(history)}\n")
print(f"  {'Style':>12}  {'Error':>10}  Move")
print(f"  {'-'*36}")
for style in STYLES:
    for err in ["Good", "Blunder"]:  
        move = predict_next_move(history, style=style, error_level=err,
                                  temperature=TEMPS[err], enforce_legal=True)
        print(f"  {style:>12}  {err:>10}  {move}")
    print()


In [ ]:
import random

ELO_PROFILE = {
    "1500": (0.58, 0.21, 0.13, 0.08),
    "1600": (0.65, 0.20, 0.10, 0.05),
    "1700": (0.72, 0.17, 0.07, 0.04),
    "1800": (0.78, 0.14, 0.05, 0.03),
}
ERROR_NAMES = ["Good", "Inaccuracy", "Mistake", "Blunder"]
TEMPS       = {"Good": 0.1, "Inaccuracy": 0.8, "Mistake": 1.2, "Blunder": 1.5}

# Style → temperature nudge
STYLE_TEMP_NUDGE = {
    "Tactical":   +0.2,
    "Aggressive": +0.3,
    "Solid":      -0.1,
    "Defensive":  -0.2,
    "Balanced":    0.0,
}

def sample_error_level(target_elo: int = 1600, phase: str = "Middlegame") -> str:
    key   = str(round(target_elo, -2))
    key   = key if key in ELO_PROFILE else "1600"
    probs = list(ELO_PROFILE[key])
    if phase == "Middlegame":
        probs[0] -= 0.03
        probs[2] += 0.02
        probs[3] += 0.01
    total = sum(probs)
    probs = [p / total for p in probs]
    return random.choices(ERROR_NAMES, weights=probs, k=1)[0]

def infer_phase(move_history: list) -> str:
    n = len(move_history)
    if n < 10:  return "Opening"
    if n > 40:  return "Endgame"
    return "Middlegame"


def trainer_bot_move(
    move_history: list,
    target_elo:   int = 1600,
    style:        str = "Balanced",  
) -> str:
    """
    Full trainer-bot: sample error level by ELO → generate conditioned move.

    style controls *how* the bot plays:
      Tactical   → sharp, combinational, volatile eval swings
      Aggressive → attacks quickly, may sacrifice material unsoundly
      Solid      → avoids complications, prefers safe moves
      Defensive  → passive, prophylactic, rarely blunders but rarely wins
      Balanced   → average club player behaviour
    """
    phase       = infer_phase(move_history)
    error_level = sample_error_level(target_elo, phase)
    temperature = max(TEMPS[error_level] + STYLE_TEMP_NUDGE.get(style, 0.0), 0.1) 
    # Main making predictions 
    move = predict_next_move(
        move_history,
        phase=phase,
        style=style,
        error_level=error_level,
        temperature=temperature,
        enforce_legal=False,
    )
    print(f"  [Bot] Style={style:>10} Phase={phase:>10} Error={error_level:>10} T={temperature:.1f}  →  {move}")
    return move


#Simulation: compare Tactical vs Defensive bot on same opening 
opening = ["e2e4", "e7e5", "g1f3", "b8c6", "f1b5"]

for bot_style in ["Tactical", "Defensive"]:
    print("=" * 62)
    print(f"  Bot style: {bot_style}  (target ELO = 1600)")
    print("=" * 62)
    game_history = ["e2e4", "e7e5", "g1f3", "b8c6", "f1b5"]  

    for _ in range(6):
        # Black move (the styled bot)
        black_move = trainer_bot_move(game_history, target_elo=1600, style=bot_style)
        if not black_move:
            print("  [Bot] No valid move"); break
        game_history.append(black_move)

        # White responds
        white_move = predict_next_move(
            game_history, style="Balanced", error_level="Good",
            temperature=0.3, enforce_legal=True
        )
        if not white_move:
            print("  [White] No valid move"); break
        game_history.append(white_move)
        print(f"  [White] {white_move}")

        print(f"  Game: {' '.join(game_history)}\n")

In [ ]:
import math
import json
import random
from collections import defaultdict
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

EVAL_SAMPLES  = 500
QUALITY_SAMPS = 150
BATCH_SIZE    = 16
random.seed(42)

def sequence_to_prompt_and_target(seq_text: str):
    """
    seq_text format:  '<ELO:…> <PHASE:…> <ERR:…> <|moves|> e2e4 c7c5 <|target|> g1f3'
    Returns (prompt_up_to_and_including_<|target|>, target_move)
    """
    sep = " <|target|> "
    if sep not in seq_text:
        return None, None
    prompt_part, target = seq_text.rsplit(sep, 1)
    prompt = prompt_part + " <|target|>"   
    return prompt, target.strip().split()[0]


def generate_move_from_prompt(prompt: str, temperature: float = 0.8) -> str:
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=(temperature > 0.01),
            temperature=max(temperature, 0.01),
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_toks  = output[0][input_length:]
    generated = tokenizer.decode(new_toks, skip_special_tokens=True).strip()
    
    for length in (5, 4):
        candidate = generated[:length].strip()
        if is_valid_uci(candidate):
            return candidate

    # Find for first whitespace-separated token that is valid UCI
    for token in generated.split():
        if is_valid_uci(token):
            return token

    return ""


In [ ]:

# 2. Move accuracy 
print("\n" + "="*60)
print(f"2. MOVE ACCURACY  (n={EVAL_SAMPLES})")
print("="*60)

sample_seqs         = random.sample(val_texts, min(EVAL_SAMPLES, len(val_texts)))
correct             = 0
invalid             = 0
attempted           = 0
correct_by_quality  = defaultdict(int)
total_by_quality    = defaultdict(int)

for seq in sample_seqs:
    prompt, target = sequence_to_prompt_and_target(seq)  
    if prompt is None:
        continue

    quality = [t for t in prompt.split() if t.startswith("<ERR:")]
    quality = quality[0].replace("<ERR:", "").replace(">", "") if quality else "Unknown"

    predicted  = generate_move_from_prompt(prompt, temperature=0.5)
    attempted += 1
    total_by_quality[quality] += 1

    if predicted == "":
        invalid += 1
    elif predicted == target:
        correct += 1
        correct_by_quality[quality] += 1

accuracy     = correct / attempted * 100 if attempted else 0
invalid_rate = invalid / attempted * 100 if attempted else 0

print(f"  Attempted      : {attempted}")
print(f"  Exact matches  : {correct}  ({accuracy:.1f}%)")
print(f"  Invalid UCI    : {invalid}  ({invalid_rate:.1f}%)")
print()
print(f"  Accuracy by quality label:")
for q in ["Good", "Inaccuracy", "Mistake", "Blunder"]:
    total = total_by_quality[q]
    acc   = correct_by_quality[q] / total * 100 if total else 0
    print(f"    {q:>12} : {correct_by_quality[q]:>4}/{total:<4}  ({acc:.1f}%)")
print(f"  (Target: Good > 15%  |  Blunder accuracy intentionally lower)")

# ── 3. Legal move rate ────────────────────────────────────────────────────────
print("\n" + "="*60)
print(f"3. LEGAL MOVE RATE  (n={EVAL_SAMPLES})")
print("="*60)

legal   = 0
illegal = 0
invalid = 0

for seq in sample_seqs:
    prompt, _ = sequence_to_prompt_and_target(seq)
    if prompt is None:
        continue

    try:
        moves_part   = prompt.split("<|moves|>")[1].replace("<|target|>", "").strip()
        move_history = moves_part.split()
        board        = board_from_history(move_history)
    except Exception:
        continue

    predicted = generate_move_from_prompt(prompt, temperature=0.8)
    if predicted == "":
        invalid += 1
        continue
    try:
        if chess.Move.from_uci(predicted) in board.legal_moves:
            legal += 1
        else:
            illegal += 1
    except Exception:
        invalid += 1

total_legal  = legal + illegal + invalid
legal_pct    = legal   / total_legal * 100 if total_legal else 0
illgl_pct    = illegal / total_legal * 100 if total_legal else 0

print(f"  Legal          : {legal}   ({legal_pct:.1f}%)")
print(f"  Illegal        : {illegal}  ({illgl_pct:.1f}%)")
print(f"  Invalid UCI    : {invalid}")
print(f"  (Target: > 85% legal with enforce_legal=False)")


# ── Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"  {'Metric':<30} {'Value':>10}  {'Target'}")
print(f"  {'-'*60}")
print(f"  {'Move Accuracy (Good)':<30} {correct_by_quality['Good'] / max(total_by_quality['Good'],1) * 100:>9.1f}%  > 15%")
print(f"  {'Legal Move Rate':<30} {legal_pct:>9.1f}%  > 85%")

results = {
    "move_accuracy":   round(accuracy, 2),
    "legal_rate":      round(legal_pct, 2),
    "invalid_rate":    round(invalid_rate, 2),
}